# Adversarial Attacks on Differential Privacy Models

This notebook demonstrates adversarial attack techniques on models trained with differential privacy using Opacus. We explore various attack methods including FGSM (Fast Gradient Sign Method) and PGD (Projected Gradient Descent) to evaluate model robustness.

In [ ]:
from advsecurenet.computer_vision.image_classification.attacks.gradient_based import FGSM, PGD
from advsecurenet.shared.types.configs.attack_configs import (
    FgsmAttackConfig,
    PgdAttackConfig,
)
from advsecurenet.shared.types.configs.attack_configs.attacker_config import (
    AttackerConfig,
)
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.computer_vision.image_classification.attacks.attacker import Attacker
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.shared.types.configs.preprocess_config import (
    PreprocessConfig,
    PreprocessStep,
)
from advsecurenet.shared.types.configs.device_config import DeviceConfig
from advsecurenet.utils.adversarial_target_generator import AdversarialTargetGenerator
from advsecurenet.datasets.targeted_adv_dataset import AdversarialDataset

from tqdm.auto import tqdm

import numpy as np

from matplotlib import pyplot as plt

## Model Preparation

Opacus differential privacy training adds module prefixes that must be removed before attacking. This preprocessing step ensures compatibility with the adversarial attack framework.

In [ ]:
import torch
import torchvision.models as models
import os

def prepare_opacus_model_for_attacks(model_path):
    """
    Load and prepare an Opacus-trained model for adversarial attacks.
    Handles prefix stripping and proper model initialization.
    
    Args:
        model_path: Path to the saved Opacus model
    
    Returns:
        Loaded and prepared model ready for attacks
    """
    
    def strip_model_prefixes(state_dict):
        """Strip both 'model.' and '_module.' prefixes from state dict keys."""
        cleaned_state_dict = {}
        for key, value in state_dict.items():
            new_key = key
            
            if new_key.startswith('_module.'):
                new_key = new_key[8:]  # Remove '_module.' prefix
            
            if new_key.startswith('model.'):
                new_key = new_key[6:]  # Remove 'model.' prefix
            
            cleaned_state_dict[new_key] = value
        
        print(f"Cleaned prefixes from {len(cleaned_state_dict)} parameters")
        return cleaned_state_dict
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    print(f"Loading Opacus-trained model from: {model_path}")
    
    checkpoint = torch.load(model_path, map_location='cpu')
    
    # Create a fresh ResNet18 model with 10 classes for CIFAR-10
    model = models.resnet18(pretrained=False, num_classes=10)
    
    cleaned_state_dict = strip_model_prefixes(checkpoint)
    
    try:
        model.load_state_dict(cleaned_state_dict, strict=False)
        print("Model loaded successfully (missing BN stats will be recomputed)")
    except Exception as e:
        print(f"Error loading model: {e}")
        raise
    
    model.eval()
    
    print(f"\nDetailed Final Layer Analysis:")
    print("=" * 50)
    
    if hasattr(model, 'fc'):
        fc_layer = model.fc
        print(f"Final layer type: {type(fc_layer).__name__}")
        print(f"Input features: {fc_layer.in_features}")
        print(f"Output features (classes): {fc_layer.out_features}")
        print(f"Weight shape: {fc_layer.weight.shape}")
        print(f"Bias shape: {fc_layer.bias.shape if fc_layer.bias is not None else 'None'}")
        
        fc_keys_in_checkpoint = [k for k in cleaned_state_dict.keys() if 'fc' in k]
        print(f"\nFC-related keys loaded from checkpoint:")
        for key in fc_keys_in_checkpoint:
            tensor_shape = cleaned_state_dict[key].shape
            print(f"   {key}: {tensor_shape}")
            
            if key.endswith('fc.weight'):
                loaded_classes = tensor_shape[0]
                print(f"   Classes detected in checkpoint: {loaded_classes}")
        
        if fc_layer.out_features == 10:
            print(f"\nFinal layer correctly has {fc_layer.out_features} classes for CIFAR-10")
        else:
            print(f"\nWARNING: Final layer has {fc_layer.out_features} classes instead of 10")
    else:
        print("No 'fc' layer found in model!")
    
    print(f"\nModel Summary:")
    print("=" * 30)
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    return model

model_path = 'experiments_files/resnet18_cifar10_baseline.pth'
model = prepare_opacus_model_for_attacks(model_path)

print("\nModel ready for adversarial attacks!")

In [ ]:
# Define preprocessing configuration for CIFAR-10 dataset
preprocess_config = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(
            name="ToDtype", params={"dtype": "torch.float32", "scale": True}
        ),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        ),
    ]
)

# Load CIFAR-10 dataset with preprocessing
dataset = DatasetFactory.load_dataset(
    dataset_name="cifar10", preprocessing=preprocess_config)
test_data = dataset["train"]

In [ ]:
# Create data loader for batch processing
dataloader = DataLoaderFactory.create_dataloader(dataset=test_data, batch_size=32)

## Fast Gradient Sign Method (FGSM) Attack

FGSM is a simple and fast adversarial attack that generates adversarial examples by adding perturbations in the direction of the sign of the gradient. This untargeted attack aims to cause misclassification without specifying a target class.

In [ ]:
# Configure device and FGSM attack parameters
device = DeviceConfig(processor="mps")

fgsm_config = FgsmAttackConfig(
    targeted=False,
    epsilon=0.1,  # Perturbation magnitude
    device=device,
)

# Initialize FGSM attack
attack = FGSM(config=fgsm_config)

In [ ]:
# Configure and initialize the attacker
attacker_config = AttackerConfig(
    model=model,
    attack=attack,
    dataloader=dataloader,
    device=device,
    return_adversarial_images=True,
)

attacker = Attacker(config=attacker_config)

In [ ]:
# Execute FGSM attack
fgsm_attack = attacker.execute()

## Projected Gradient Descent (PGD) Attack

PGD is an iterative adversarial attack that applies multiple small perturbations, projecting back to the allowed perturbation space after each step. It is typically more effective than FGSM as it can find stronger adversarial examples through multiple iterations.

In [ ]:
# Configure device and PGD attack parameters
device = DeviceConfig(processor="mps")

pgd_config = PgdAttackConfig(
    targeted=False,
    epsilon=0.1,  # Maximum perturbation magnitude
    device=device,
)

# Initialize PGD attack
attack = PGD(config=pgd_config)

# Configure and initialize the attacker for PGD
attacker_config = AttackerConfig(
    model=model,
    attack=attack,
    dataloader=dataloader,
    device=device,
    return_adversarial_images=True,
)

attacker = Attacker(config=attacker_config)

In [ ]:
# Execute PGD attack
pgd_attack = attacker.execute()

## Results Analysis

The attack results contain information about success rates, adversarial examples, and perturbation statistics. Compare the effectiveness of FGSM vs PGD attacks on the differential privacy-trained model to evaluate robustness properties.